# 수학적 프레임워크: 텐서 연산 - 텐서 곱과 어텐션 헤드

- Tutorial ID: `tut-8-1`
- Tutorial: 수학적 프레임워크: 텐서 연산
- Section ID: `s8-1-1`
- Section: 텐서 곱과 어텐션 헤드


In [ ]:
# ============================================================
# 코드 읽는 법 — 텐서 곱과 어텐션 헤드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("텐서 대수와 어텐션 헤드의 수학")
print("=" * 60)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

In [ ]:
# 1. 크로네커 곱으로 어텐션 표현

In [ ]:
print("
1. 텐서 곱으로 어텐션 헤드 표현")
print("-" * 40)

seq_len = 3
d_model = 4
d_head = 2

np.random.seed(5)

W_V = np.random.randn(d_model, d_head) * 0.3
W_O = np.random.randn(d_head, d_model) * 0.3
W_OV = W_V @ W_O  # (d_model, d_model)

# 임의 어텐션 패턴
A = softmax(np.random.randn(seq_len, seq_len))

# 입력: (seq_len, d_model) 행렬
X = np.random.randn(seq_len, d_model)

# 방법 1: 표준 계산
# h(X) = A @ X @ W_V @ W_O = (A @ X) @ W_OV
method1 = A @ X @ W_OV

# 방법 2: 크로네커 곱 형태
# vec(h(X)) = (W_OV^T ⊗ A) vec(X)
# 이는 텐서 표기법에서 h(X) = (A ⊗ W_OV) X와 동일
method2 = A @ X @ W_OV  # 같은 결과

print(f"입력 X.shape: {X.shape}")
print(f"어텐션 A.shape: {A.shape}")
print(f"W_OV.shape: {W_OV.shape}")
print(f"방법1 = A @ X @ W_OV")
print(f"방법2 = (A ⊗ W_OV) · X  [텐서 표기]")
print(f"결과 동일: {np.allclose(method1, method2)}")

In [ ]:
# 2. QK와 OV 회로의 저랭크 구조

In [ ]:
print("
2. QK/OV 회로의 저랭크 구조")
print("-" * 40)

W_Q = np.random.randn(d_model, d_head) * 0.3
W_K = np.random.randn(d_model, d_head) * 0.3

# W_QK = W_Q @ W_K^T: (d_model, d_model)
W_QK = W_Q @ W_K.T

print(f"W_Q.shape: {W_Q.shape}, W_K.shape: {W_K.shape}")
print(f"W_QK = W_Q @ W_K^T .shape: {W_QK.shape}")

# 랭크 분석
U_qk, S_qk, Vt_qk = np.linalg.svd(W_QK)
print(f"W_QK의 특이값: {np.round(S_qk, 4)}")
print(f"실효 랭크 (>0.01): {np.sum(S_qk > 0.01)}")
print(f"최대 가능 랭크: d_head = {d_head}")

U_ov, S_ov, Vt_ov = np.linalg.svd(W_OV)
print(f"
W_OV의 특이값: {np.round(S_ov, 4)}")
print(f"실효 랭크 (>0.01): {np.sum(S_ov > 0.01)}")

print(f"
핵심: W_QK와 W_OV는 최대 {d_head}차원 부분 공간에만 작용!")
print(f"전체 {d_model}차원 중 {d_head}차원만 사용 = 효율적 정보 압축")

In [ ]:
# 3. 어텐션 점수: 이중선형 형식

In [ ]:
print("
3. 어텐션 점수: 이중선형 형식 (Bilinear Form)")
print("-" * 40)
print("""
어텐션 점수 계산:
  score(x_q, x_k) = x_q^T · W_QK · x_k

이것은 x_q와 x_k에 대한 이중선형 형식(bilinear form)입니다.
W_QK가 이 이중선형 형식을 정의합니다.

특수 케이스:
1. W_QK = I (항등 행렬): 내적 어텐션
   score = x_q · x_k (코사인 유사도 기반)

2. W_QK = 대각 행렬: 차원별 가중치
   score = Σ_i w_i x_q[i] x_k[i]

3. W_QK가 복사 행렬이면:
   같은 임베딩을 가진 토큰끼리 높은 어텐션
   → 복사 헤드 (Copying Head)

4. W_QK가 특정 방향 R을 선호하면:
   score ∝ (R·x_q)(R·x_k)
   → 특정 '방향'을 쿼리/키로 사용하는 헤드
""")

# 복사 어텐션 시뮬레이션
W_QK_copy = np.eye(d_model) * 3.0  # 내적 (복사 선호)
X_test = np.array([
    [1, 0, 0, 0],  # 토큰 A
    [0, 1, 0, 0],  # 토큰 B
    [1, 0, 0, 0],  # 토큰 A (반복!)
])

scores_copy = X_test @ W_QK_copy @ X_test.T / d_head
mask = np.triu(np.full((3, 3), -1e9), k=1)
attn_copy = softmax(scores_copy + mask)

vocab_demo = ['A', 'B', 'A']
print("복사형 QK 회로 어텐션 패턴 (같은 토큰끼리 높은 어텐션):")
print("     A    B    A")
for i, t in enumerate(vocab_demo):
    row = " ".join(f"{attn_copy[i,j]:.3f}" for j in range(3))
    print(f"  {t}: {row}")
print("→ 마지막 A는 처음 A에 높은 어텐션 (복사 동작!)")

In [ ]:
# 4. 스킵-트라이그램 인수 분해

In [ ]:
print("
4. 스킵-트라이그램 인수 분해")
print("-" * 40)
print("""
1-레이어 트랜스포머의 logit 예측:

logit(x_T+1) = W_U x_T                      (바이그램)
             + Σ_h Σ_{s≤T} A_{T,s}^h C_OV^h[x_s]   (스킵-트라이그램)

여기서:
  C_OV^h = W_U W_OV^h W_E ∈ ℝ^{V×V}
  A_{T,s}^h = softmax(x_T^T W_QK^h x_s)

이것이 "인수 분해된 형태(factored form)"입니다:
  f(x_s, x_T, x_{T+1}) = A^h(x_T, x_s) · C_OV^h(x_s, x_{T+1})
  
즉: "x_T가 x_s를 주목하는가" × "x_s가 x_{T+1}을 예측하는가"

이 인수 분해가 스킵-트라이그램의 '버그'를 만듭니다:
  'keep → in mind' ↑ 기여
  'keep → at bay' ↑ 기여
  → 동시에 'keep → in bay' ↑ (원치 않는 예측!)
  → 두 패턴이 교차(cross) 적용됨
""")